<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/x-segmentation-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the course · Segmentation 1 — Train your own nucleus finder

On Friday you ran Cellpose, a model other scientists trained, and counted
the nuclei it found. In this notebook you build and train your own
segmentation model, a **U-Net**, from an empty network to masks you can score.
By the end you will have a model that outlines nuclei better than Tuesday's
threshold, and you will know every line of it.

This is an extension pathway: you work through it alone or with a partner,
at your own pace. You need Thursday's CNN notebook and Friday's
segmentation starter behind you (you should be able to build a small CNN
and compute IoU). Allow about two hours.

How it works, as in class: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check under it. A check that fails tells you what to look at. Each
section ends with a **You're done when** line.

**GPU or not?** Section 3 trains the model. With a GPU (Runtime → Change
runtime type → T4 GPU) it trains on bigger pieces for longer. Without one it
trains a shorter run, a few minutes on the free CPU. Both work; the GPU's
masks should be a little cleaner.

In [ ]:
import os
import hashlib
import shutil
import glob
import time
import pathlib
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
import tifffile
from scipy import ndimage
import torch
from torch import nn
import torch.nn.functional as F


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = device.type == "cuda"
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
np.random.seed(0)
print("Running on", device)
DATA = pathlib.Path("data/bbbc039")

## 1. The data: nuclei with hand-drawn outlines

To train a segmentation model you need images *and* the right answer for
every pixel. **BBBC039** is a public set of 200 microscope images of cell
nuclei, glowing under a blue fluorescent stain, where scientists outlined
about 23,000 nuclei by hand. It is free for any use (licence CC0).

These images look different from Project II's brown-and-blue tissue: the
nuclei are bright on black. That keeps your attention on the method. Friday's
`nuclei_channel` turns a stained tissue image into the same kind of picture
(nuclei bright, everything else dark), so all of this carries over.

Next come two cells: one downloads the images, and the next loads one and
shows it. If the download stops with an error, run its cell again.

⬇️ **Download the data (81 MB).** Only run the next cell if you are going to use this data. It downloads 81 MB (the BBBC039 images and masks), so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, so use "Run all" only when you mean to do the whole notebook.

In [ ]:
# ⬇️ Download cell: BBBC039 images and masks (81 MB)
BBBC_SHA256 = {
    "images": "6f30a5d4fe38c928ded972704f085975f8dc0d65d9aa366df00e5a9d449fddd7",
    "masks": "f9e6043d8ca56344a4886f96a700d804d6ee982f31e2b2cd3194af2a053c2710",
}


def fetch(name):
    """Download and unzip one BBBC039 file, once."""
    DATA.mkdir(parents=True, exist_ok=True)
    zpath = DATA / f"{name}.zip"
    if not fetch_from_mirror(f"https://data.tk.sg/bbbc039/{name}.zip", zpath, BBBC_SHA256[name]):
        urllib.request.urlretrieve(f"https://data.broadinstitute.org/bbbc/BBBC039/{name}.zip", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA)
    zpath.unlink()


if (DATA / "images").exists() and (DATA / "masks").exists():
    print("BBBC039 is already downloaded. Nothing to do.")
else:
    for part in ("images", "masks"):
        if not (DATA / part).exists():
            fetch(part)
    print("Done: BBBC039 downloaded (81 MB).")

In [ ]:
require_data("BBBC039", DATA / "images", DATA / "masks")
image_files = sorted(glob.glob(str(DATA / "images" / "*.tif")))
print(len(image_files), "images")


def load_image(path):
    """Read a 16-bit image and scale it to 0–1 (the brightest 0.5% become 1)."""
    img = tifffile.imread(path).astype(np.float32)
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((img - lo) / (hi - lo), 0, 1).astype(np.float32)


def load_colours(path):
    """The hand-drawn mask for an image, as stored: 0 = background, 1, 2, 3 = nucleus colours."""
    mask_path = path.replace("images", "masks").replace(".tif", ".png")
    return iio.imread(mask_path)[:, :, 0]


img = load_image(image_files[0])
colours = load_colours(image_files[0])
print("image", img.shape, img.dtype, "  mask values:", np.unique(colours))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("image")
axes[1].imshow(colours, cmap="viridis", interpolation="nearest"); axes[1].set_title("mask as stored")
for ax in axes:
    ax.axis("off")
plt.show()

The mask uses only four values. Background is 0, and each nucleus is coloured
1, 2 or 3, chosen so that **two touching nuclei never share a colour**. That
lets the file separate touching nuclei with only three colours.

To count nuclei we want a **label image** like Friday's: 0 for background and
a different number (1, 2, 3 … up to the count) for every nucleus. Here is the
idea for colour 1 alone. `ndimage.label`, from Tuesday, numbers each separate
blob of colour 1:

In [ ]:
blobs_1, n_1 = ndimage.label(colours == 1)
print("colour 1 has", n_1, "separate nuclei; their labels run from 1 to", blobs_1.max())

🔮 **Predict.** If colour 1 has `n_1` nuclei, colour 2 has `n_2` and colour 3
has `n_3`, how many nuclei does the image have? And if you labelled each colour
separately, why would you have to change the numbers before putting the three
label images together?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 1a.** Write `to_labels(colours)`. For each colour 1, 2 and 3:
label its blobs with `ndimage.label`, add how many nuclei you have already
numbered (`total`) so the numbers don't repeat, and write them into `labels`
where that colour's blobs are. Return the label image.

In [ ]:
def to_labels(colours):
    labels = np.zeros(colours.shape, dtype=int)
    total = 0
    for colour in (1, 2, 3):
        blobs, n = ndimage.label(colours == colour)
        ...  # put blobs + total into labels, where blobs > 0
        ...  # then add n to total
    return labels

In [ ]:
demo = np.array([[1, 1, 0, 2],
                 [0, 0, 0, 2],
                 [3, 0, 1, 0]])
out = to_labels(demo)
check(out.max() > 0, "to_labels returned an empty label image. Did you write into `labels`?")
check(len(np.unique(out)) - 1 == 4, f"The demo has 4 nuclei (two of colour 1, one each of 2 and 3); you have {len(np.unique(out)) - 1} different labels.")
check(out.max() == 4, f"Labels should run 1 to 4, but the largest is {out.max()}. Did you add `total` and then grow it by n?")
check(out[0, 0] == out[0, 1] and out[0, 0] != out[2, 2], "The two pixels of one nucleus share a label, and different nuclei get different labels.")
labels = to_labels(colours)
print("nuclei in the first image:", labels.max())

Everyone continues from here (code hidden; double-click to see it after).

In [ ]:
#@title Run this to load all 200 images { display-mode: "form" }
def to_labels(colours):
    labels = np.zeros(colours.shape, dtype=int)
    total = 0
    for colour in (1, 2, 3):
        blobs, n = ndimage.label(colours == colour)
        where = blobs > 0
        labels[where] = total + blobs[where]
        total = total + n
    return labels


images = [load_image(f) for f in image_files]
label_images = [to_labels(load_colours(f)) for f in image_files]
# A fixed split: the first 150 images to train on, the last 50 to judge with.
train_imgs, train_labs = images[:150], label_images[:150]
val_imgs, val_labs = images[150:], label_images[150:]
if QUICK:
    val_imgs, val_labs = val_imgs[:4], val_labs[:4]
print(len(train_imgs), "training images,", len(val_imgs), "validation images")
print("nuclei in the training images:", sum(lab.max() for lab in train_labs))

**You're done when** your `to_labels` check passes and the cell above reports
150 training images.

## 2. The U-Net's shape: down, across, up

A classifier like Thursday's CNN shrinks an image step by step until only a
few numbers are left: "cat" or "dog". Segmentation needs an answer **for every
pixel**, so the output must be as big as the input.

A **U-Net** does both. Its left half is a CNN going *down*: convolutions,
then max-pooling to halve the size, so later layers see a wider area. Its
right half goes back *up*: a **transposed convolution** (`ConvTranspose2d`)
doubles the size again. Drawn on paper, it is the shape of a U.

Going down throws away *where* things are exactly. So at each level the U-Net
copies the left side's feature maps straight across and joins them to the
right side's: a **skip connection**. The right side then has both "this is a
nucleus" from below and "its edge is exactly here" from across.
`torch.cat([a, b], dim=1)` stacks two feature maps along the channel
dimension, so 32 channels joined to 32 channels makes 64.

Here are the building blocks. `block` is two convolutions with padding 1, so
it keeps the height and width (Thursday's shape rule).

In [ ]:
def block(c_in, c_out):
    """Two 3×3 convolutions that keep the height and width."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
    )


x = torch.zeros(1, 1, 128, 128)                    # one greyscale 128×128 crop
down = block(1, 16)(x)
print("after block(1, 16):     ", tuple(down.shape))
print("after max_pool2d(2):    ", tuple(F.max_pool2d(down, 2).shape))
up = nn.ConvTranspose2d(16, 8, 2, stride=2)(F.max_pool2d(down, 2))
print("after ConvTranspose2d:  ", tuple(up.shape))

🔮 **Predict.** Our U-Net goes down twice: 128 → 64 → 32, with 16, 32 and 64
channels. At the bottom, what is the shape of the feature map for one 128×128
crop? Coming back up to 64×64, the transposed convolution gives 32 channels and
the skip connection brings 32 more. How many channels go into the next block?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** The down path and the first step up are written. Finish
the second step up in `forward`: apply `self.up1` to `y`, join the result with
`x1` (the full-size feature map from the very first block), and pass the join
through `self.join1`. Copy the line above it and change three names.

In [ ]:
class UNet(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.down1 = block(1, c)          # 128×128, c channels
        self.down2 = block(c, 2 * c)      # 64×64, 2c
        self.down3 = block(2 * c, 4 * c)  # 32×32, 4c: the bottom of the U
        self.up2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2)
        self.join2 = block(4 * c, 2 * c)  # 2c from up2 + 2c from down2
        self.up1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2)
        self.join1 = block(2 * c, c)      # c from up1 + c from down1
        self.head = nn.Conv2d(c, 1, 1)    # one score per pixel: "nucleus?"

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(F.max_pool2d(x1, 2))
        x3 = self.down3(F.max_pool2d(x2, 2))
        y = self.join2(torch.cat([self.up2(x3), x2], dim=1))
        y = ...
        return self.head(y)

In [ ]:
net = UNet()
try:
    out = net(torch.zeros(2, 1, 64, 64))
except TypeError:
    out = ...
except RuntimeError:
    raise AssertionError("Not yet. A layer got the wrong number of channels or the wrong size. "
                         "Join self.up1(y) with x1 (the full-size map from down1), then pass the join to self.join1.")
check(is_done(out), "forward still has a `...`. Finish the line for the second step up.")
check(tuple(out.shape) == (2, 1, 64, 64), f"The output should be the input's size, (2, 1, 64, 64), but it is {tuple(out.shape)}.")
n_params = sum(p.numel() for p in net.parameters())
print(f"Your U-Net has {n_params:,} numbers to learn.")

Everyone continues from here (code hidden; double-click to see it after).

In [ ]:
#@title Run this to continue { display-mode: "form" }
class UNet(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.down1, self.down2, self.down3 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c)
        self.up2, self.join2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.join1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.head = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(F.max_pool2d(x1, 2))
        x3 = self.down3(F.max_pool2d(x2, 2))
        y = self.join2(torch.cat([self.up2(x3), x2], dim=1))
        y = self.join1(torch.cat((self.up1(y), x1), 1))
        return self.head(y)

🐛 **Break it.** Change `torch.zeros(2, 1, 64, 64)` below to a 66×66 input
and run it. Read the error. Which line of `forward` failed, and why does a
U-Net that halves twice need a height and width divisible by 4? (BBBC039's
images are 520×696. Are they?)

In [ ]:
print(tuple(UNet()(torch.zeros(2, 1, 64, 64)).shape))

**You're done when** your U-Net's output has the same height and width as its
input, and you can say what the skip connections carry across.

## 3. Train it

Training is Wednesday's five-line step. Two things are new:

- **The batches are random crops.** Each step cuts small squares from random
  training images, with the matching squares of their label images. It is
  Tuesday's tiling, at random places. A crop is faster to train on than a
  whole image, and a nucleus looks the same anywhere in the picture.
- **The loss is per pixel.** The model gives every pixel a raw score: any
  number, very negative for "surely background" and very positive for
  "surely nucleus". Raw scores like these are called **logits**.
  `binary_cross_entropy_with_logits` turns each score into a probability
  from 0 to 1 (with the sigmoid function, which section 4 shows you),
  compares it with the answer at that pixel (1 = nucleus, 0 = background)
  and averages over all pixels: cross-entropy, as on Wednesday, for a
  yes-or-no question asked at every pixel.

Here is how to cut one crop at a fixed place, the way Friday's `get_tile`
did:

In [ ]:
r, c, size = 100, 200, 128
crop = train_imgs[0][r:r + size, c:c + size]
crop_answer = train_labs[0][r:r + size, c:c + size] > 0    # True on nuclei
print(crop.shape, crop_answer.shape, f"{crop_answer.mean():.0%} of this crop is nucleus")

✏️ **Exercise 3a.** Write `random_crop(img, lab, size)`: choose a random top
row `r` from 0 up to and including `img.shape[0] - size` (the last row where
a crop still fits), and a random left column `c` the same way with
`img.shape[1]`. `np.random.randint(high)` gives a whole number from 0 to
`high - 1`, so use `np.random.randint(img.shape[0] - size + 1)`. Return the
crop of `img` and the crop of `lab > 0`.

In [ ]:
def random_crop(img, lab, size):
    r = ...
    c = ...
    return ..., ...

In [ ]:
grid = np.arange(20 * 30).reshape(20, 30).astype(np.float32)
pieces = [random_crop(grid, grid, 8) for _ in range(200)]
check(all(is_done(a, b) for a, b in pieces), "random_crop still returns `...`.")
check(all(a.shape == (8, 8) and b.shape == (8, 8) for a, b in pieces), "Both crops should be size × size. Did you slice from r to r + size?")
check(all(np.array_equal(a > 0, b) for a, b in pieces), "The second crop must come from the same place as the first, as True/False (`lab > 0`).")
corners = {int(a[0, 0]) for a, _ in pieces}
check(len(corners) > 50, "The crops all start in a few places. Pick r and c at random each time.")
check(max(int(a[-1, -1]) for a, _ in pieces) <= grid.max(), "A crop runs off the edge.")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def random_crop(img, lab, size):
    top = np.random.randint(0, img.shape[0] - size + 1)
    left = np.random.randint(0, img.shape[1] - size + 1)
    window = (slice(top, top + size), slice(left, left + size))
    return img[window], lab[window] > 0


def random_batch(n, size):
    pairs = [random_crop(train_imgs[k], train_labs[k], size) for k in np.random.randint(len(train_imgs), size=n)]
    x = torch.tensor(np.array([p[0] for p in pairs]))[:, None]
    y = torch.tensor(np.array([p[1] for p in pairs]), dtype=torch.float32)[:, None]
    return x.to(device), y.to(device)

🔮 **Predict.** Across all the training images, about 15% of pixels are
nucleus (the crop above happened to have more). A model that has
learned nothing outputs a probability of 0.5 at every pixel, and the loss for
that is about 0.69. What could the model do to get the loss lower fast, in
the first few dozen steps, without finding a single nucleus?

In [ ]:
my_prediction = ""  # @param {type:"string"}

Now train. With a GPU this does 1,500 steps on 32 crops of 256×256; without
one, 250 steps on 8 crops of 128×128. Each step prints nothing; every 50
steps you see the loss and the time so far.

In [ ]:
STEPS = 5 if QUICK else (1500 if GPU else 250)
BATCH, SIZE = (32, 256) if GPU else (8, 128)

model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
start = time.time()
model.train()
for step in range(STEPS):
    x, y = random_batch(BATCH, SIZE)
    optimizer.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {loss.item():.3f}  {time.time() - start:.0f} s")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("step"); plt.ylabel("training loss"); plt.yscale("log")
plt.show()

**You're done when** the loss has gone below about 0.15 (CPU) or 0.08 (GPU)
and is still falling slowly at the end.

## 4. Score it

The model gives a score per pixel. `torch.sigmoid` turns a score into a
probability from 0 to 1, and a **threshold** turns probabilities into a
True/False mask. Here is the model on one whole validation image. The U-Net
has no fixed input size, so it can run on the full 520×696 image at once.

In [ ]:
model.eval()
with torch.no_grad():
    scores = model(torch.tensor(val_imgs[0])[None, None].to(device))[0, 0].cpu()
probability = torch.sigmoid(scores).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(val_imgs[0], cmap="gray"); axes[0].set_title("validation image")
axes[1].imshow(probability, cmap="magma", vmin=0, vmax=1); axes[1].set_title("probability of nucleus")
axes[2].imshow(val_labs[0] > 0, cmap="gray"); axes[2].set_title("hand-drawn answer")
for ax in axes:
    ax.axis("off")
plt.show()

✏️ **Exercise 4a.** Write `predict_mask(model, img, threshold)`. Run the model
on the image as above, apply `torch.sigmoid`, and return a NumPy True/False
array that is True where the probability is above `threshold`.

In [ ]:
def predict_mask(model, img, threshold=0.5):
    model.eval()
    with torch.no_grad():
        scores = ...
        return ...

In [ ]:
mask = predict_mask(model, val_imgs[0])
check(is_done(mask), "predict_mask still returns `...`.")
check(isinstance(mask, np.ndarray) and mask.dtype == bool, "Return a NumPy array of True/False. Compare with `> threshold`, then `.numpy()`.")
check(mask.shape == val_imgs[0].shape, f"The mask should be the image's shape {val_imgs[0].shape}, not {mask.shape}.")
check(predict_mask(model, val_imgs[0], 0.0).mean() > predict_mask(model, val_imgs[0], 1.0).mean(), "A lower threshold should mark more pixels. Use the threshold argument.")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def predict_mask(model, img, threshold=0.5):
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(img)[None, None].to(device))
        return (torch.sigmoid(out[0, 0]).cpu() > threshold).numpy()


def iou(a, b):
    return (a & b).sum() / (a | b).sum()

Is it any good? Compare it with the best a single threshold on the image
could do. **Otsu's method** (Tuesday's extension) picks the threshold that
best separates bright from dark pixels, with no learning at all.

🔮 **Predict.** On these bright-on-black images, will your U-Net beat Otsu's
threshold by a lot, a little, or not at all? Guess both mean IoUs.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
from skimage.filters import threshold_otsu

unet_scores, otsu_scores = [], []
for img, lab in zip(val_imgs, val_labs):
    truth = lab > 0
    unet_scores.append(iou(predict_mask(model, img), truth))
    otsu_scores.append(iou(img > threshold_otsu(img), truth))
print(f"mean IoU on {len(val_imgs)} validation images:  U-Net {np.mean(unet_scores):.3f}   Otsu {np.mean(otsu_scores):.3f}")
worst = int(np.argmin(unet_scores))
print(f"the U-Net's worst image is validation image {worst}, IoU {unet_scores[worst]:.3f}")

Look at that worst image next to its answer. Is the model wrong, or is the
hand-drawn answer missing something? People make labelling mistakes too.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(val_imgs[worst], cmap="gray"); axes[0].set_title(f"validation image {worst}")
axes[1].imshow(predict_mask(model, val_imgs[worst]), cmap="gray"); axes[1].set_title("U-Net")
axes[2].imshow(val_labs[worst] > 0, cmap="gray"); axes[2].set_title("hand-drawn answer")
for ax in axes:
    ax.axis("off")
plt.show()

✏️ **Exercise 4b.** 0.5 is not always the best threshold. Try the thresholds
in `candidates` on the validation images and put the one with the highest
mean IoU in `best_threshold`. (Choosing a setting on validation data is fine.
Choosing it on the test set would be Wednesday's leak: you would have tuned on
the O-level paper.)

In [ ]:
candidates = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
best_threshold = ...
print("best threshold:", best_threshold)

In [ ]:
check(is_done(best_threshold), "Set best_threshold to one of the candidates.")
check(best_threshold in candidates, "best_threshold should be one of the numbers in `candidates`.")
score = {t: np.mean([iou(predict_mask(model, img, t), lab > 0) for img, lab in zip(val_imgs, val_labs)]) for t in candidates}
check(np.isclose(score[best_threshold], max(score.values())), f"Another threshold scores higher: {max(score, key=score.get)}. Compute the mean IoU for every candidate.")

Save your model. Colab deletes files when the runtime stops, so download it
(Files panel on the left → `unet_nuclei.pt` → ⋮ → Download) if you want to
keep it. Segmentation 2 trains its own copy, so you don't need to.

In [ ]:
torch.save(model.state_dict(), "unet_nuclei.pt")
print("saved unet_nuclei.pt")

**You're done when** you have a mean validation IoU for your U-Net and for
Otsu, a best threshold, and an opinion on the worst image.

### Stretch: make it yours

Pick one or more. Change one thing at a time and write down the IoU each time,
as in the log you kept in Thursday's competition.

- **Fewer labels.** Train on only 10 of the 150 training images. How much IoU
  do you lose? Project II teams have to draw their own masks, so this tells
  you how many to draw.
- **Dice loss.** Add a Dice loss, `1 - 2·|A∩B| / (|A| + |B|)` computed with
  probabilities instead of True/False, to the cross-entropy. It cares about the
  nucleus pixels only, so the background cannot dominate.
- **Wider or deeper.** Try `UNet(c=32)`, or add a third level. More numbers to
  learn: does validation IoU go up, or only training loss down?
- **Tissue.** Run your model on Friday's immunohistochemistry image after
  `nuclei_channel` (Friday's notebook, section 2; nuclei are already bright
  there, but resize the tile so they are about as big as here).
  `nuclei_channel` gives whole numbers from 0 to 255, and your model learned
  on pictures from 0 to 1, so divide by 255 first and make it `float32`.
  What goes wrong, and why?

In [ ]:
# Your stretch work here.